# VITS → B1 droid — Colab training

Fine-tunes VITS on the `Dmi1tr13/ljspeech-b1` droid dataset (branch `finetune`).

**Before running:** `Runtime → Change runtime type → GPU (T4/L4) → Save`.

All fixes live in the source code (held-out test set, quiet logs, `--max-train-clips`). This notebook just clones, installs, trains, and listens — no patch cells.

## 1. Clone the repo (branch with the fine-tuning fixes)

In [ ]:
!git clone -b finetune https://github.com/Dmitri1S9/DL.git project
%cd project

## 2. Install dependencies
System `espeak-ng` (the VITS tokenizer needs it) + Python deps. We skip the `torch+cu128` pin (a local GPU build unavailable on Colab) and use Colab's preinstalled torch.

In [ ]:
!apt-get -qq install -y espeak-ng
!grep -vE '^(torch|pedalboard)' requirements.txt > requirements-colab.txt
!pip install -q -r requirements-colab.txt
print('OK')

## 3. Sanity check — GPU + espeak

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!espeak-ng --version

## 4. Train
Mirrors the known-good run: batch 2, default LR (2e-4), ~2000 steps.
`--max-train-clips 4000` ÷ batch 2 = 2000 steps in one epoch; `--num-workers 4` splits the espeak
memory-map load across processes (Colab can't raise `vm.max_map_count`). The last 500 clips stay held
out for evaluation regardless.

In [ ]:
!PYTHONPATH=src python -m vits_finetune.train \
    --batch-size 2 --num-epochs 1 --max-train-clips 4000 --num-workers 4

## 5. Listen — pretrained vs fine-tuned

In [ ]:
TEXT = 'Roger roger. All units, proceed with the mission. Standing by.'
!PYTHONPATH=src python -m vits_finetune.synthesize --text "$TEXT" --out demo_pretrained.wav
!PYTHONPATH=src python -m vits_finetune.synthesize --checkpoint models/vits_finetune/epoch_1.pt --text "$TEXT" --out demo_finetuned.wav

from IPython.display import Audio, display
print('=== Pretrained (natural) ==='); display(Audio('demo_pretrained.wav'))
print('=== Fine-tuned (droid) ===');   display(Audio('demo_finetuned.wav'))

## 6. Evaluate intelligibility with Whisper (objective check)
Transcribe the fine-tuned audio and compare to the input text — low WER = intelligible, ~100% = garbage.

In [ ]:
import whisper, jiwer
asr = whisper.load_model('base')
ref = TEXT.lower().strip('.').strip()
for tag, wav in [('pretrained', 'demo_pretrained.wav'), ('finetuned', 'demo_finetuned.wav')]:
    hyp = asr.transcribe(wav)['text'].lower().strip().strip('.')
    print(f'[{tag}] WER={jiwer.wer(ref, hyp):.2%}  ->  {hyp!r}')